# 28 · Agentic RAG 与 Agentic Retrieval

> 把 RAG 从“固定流水线”升级为“**模型自主决策的循环**”：需要就检索、不够就再搜、错就纠正、够了才回答。

**本文件覆盖知识点**：Agentic RAG / Agentic Retrieval / Query·Search Planning / Tool Calling / Iterative Retrieval / Retrieval Decision / Stop Condition

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 从流水线到智能体

```text
普通 RAG:    Query → Retrieve → LLM           （一次到底）

Agentic RAG: User Query → Agent → Plan → Search
            → Evaluate → Search Again → Reason → Answer
                    ↑_________ 循环直到满意/达上限
```

智能体拥有三件套（01 课提过）：
- **Planning**：把任务拆步、决定检索还是直接答；
- **Tool Calling**：把“检索/查库/搜网页”做成工具，模型自主调用（Function Calling）；
- **Loop**：根据上一步结果决定继续还是停（Stop Condition）。

In [ ]:
# 一个“决策循环”骨架：需要就检索、够了就答
# 真实实现里，检索工具来自本课程各步骤，这里用桩函数演示控制流

class MiniAgentRAG:
    """迷你 Agent 化 RAG：模型每步可选 retrieve / answer / stop"""
    def __init__(self):
        self.max_steps = 3   # 兜底：最多检索几次，防止死循环
        self.collected = []

    def retrieve(self, q):
        """桩：模拟一次检索（生产接真实检索器）"""
        return ['（检索到）' + q + ' 的相关文档片段']

    def decide(self, ctx):
        """桩：模拟 LLM 决策——是否已有足够信息作答（生产让 qwen 输出结构化决策）"""
        # 这里简化：跑满 max_steps 前总说“继续检索”（演示循环）；真实应看 eval 判据
        return 'retrieve' if len(self.collected) < self.max_steps else 'answer'

    def answer(self, q):
        """桩：基于收集到的上下文生成"""
        return '最终答案：' + '；'.join(self.collected)

    def run(self, q):
        steps = 0
        while steps < self.max_steps:
            steps += 1
            action = self.decide(self.collected)          # 检索决策
            print(f'step{steps}: 决策={action}')
            if action == 'answer':
                return self.answer(q)                      # 停止条件达成
            self.collected += self.retrieve(q)             # 迭代检索
        return self.answer(q)                              # 达上限强制收尾

rag = MiniAgentRAG()
print(rag.run('星云机器人支持私有化吗'))

In [ ]:
# 知识点·真调说明：Retrieval Decision / Tool 调用意图 —— 真调模型在“要不要检索、调哪个工具”上表态
import json as _json

def _decide(qu, fb):
    out = _llm_live(
        prompt='用户提问：' + qu,
        system='你是 Agentic RAG 的决策路由器。只输出一个 JSON：'
               '若需检索知识库（涉及私有或未见过的资料）：{"action": "retrieve", "query": "<改写后的检索词>", "reason": "为什么检索"}；'
               '若常识可直接回答：{"action": "answer", "answer": "<直接回答>", "reason": "为什么不用检索"}。禁止输出其它文字。',
        fallback=fb,
        temperature=0.1,
    )
    if out is None:
        out = fb
        print('（以上为固定样例；下面按样例解析）')
    try:
        return _json.loads(out)
    except Exception as _e:
        print('解析失败：', _e)
        return None

print('① 涉及私有产品资料 → 该走 retrieve')
_d1 = _decide('星云企业版私有化部署的最低硬件要求是多少？',
              '{"action": "retrieve", "query": "星云企业版 私有化部署 硬件要求", "reason": "私有化参数属产品私有资料，需查库"}')
if _d1:
    print('   决策 action=%s | 检索词：%s' % (_d1['action'], _d1['query']))
print()
print('② 常识问题 → 该走 answer（Stop Condition：无需检索即可停）')
_d2 = _decide('什么是大语言模型？',
              '{"action": "answer", "answer": "大语言模型是基于海量文本训练、能理解并生成文本的深度学习模型。", "reason": "公开常识，无需检索"}')
if _d2:
    print('   决策 action=%s' % _d2['action'])
print()
print('→ 同一个“决策接口”，不同 query 走出 retrieve / answer 两条路——'
      '这就是 Agentic RAG 的 Retrieval Decision 与 Stop Condition；把上面桩 decide() 换成这里真调即可。')

## 2. Agentic Retrieval 的关键设计点

| 设计 | 说明 |
|------|------|
| **Query/Search Planning** | 规划要搜什么、搜几次、每步换什么工具 |
| **Iterative Retrieval** | 多轮检索，每轮把“还缺什么”再搜一次 |
| **Self-Reflection / Correction** | 模型自评当前信息是否够、是否跑偏 |
| **Stop Condition** | 信息足够/步数上限/检索无增益 → 停 |
| **工具** | Tool Calling：检索器、SQL、网页搜索都是工具 |



In [ ]:
# 知识点·真调说明：Query / Search Planning —— 让模型把宽问题拆成“先搜什么、后搜什么”的检索计划
_q = '星云产品支持私有化部署吗？如果要私有化，官方给的最低硬件配置和年费大概是多少？'
print('宽问题：', _q)
print()
_llm_live(
    prompt=_q + '\n请为回答它规划检索步骤：需要搜几次、每步用什么关键词、查哪类资料。',
    system='你是 Agentic RAG 的查询规划器。先判断该问含哪几个子问题，再按依赖顺序给出检索计划。'
           '输出格式（不要多余解释，每行一条）：\n1) 子问题概述 —— 建议检索词\n2) …',
    fallback='未配置 Key 的固定样例：\n'
             '1) 星云是否支持私有化部署 —— 检索词：星云 私有化 部署方式\n'
             '2) 私有化最低硬件配置 —— 检索词：星云 私有化 硬件要求\n'
             '3) 私有化年费/授权 —— 检索词：星云 私有化 价格 授权',
    temperature=0.2,
)
print('→ 一条用户问题被拆成“分步检索计划”，Agent 按计划逐轮调用检索工具并自评“够不够”——这就是 Agentic Retrieval 的 Search Planning。')

## 3. 家族成员（下一课细看）

- **CRAG**：检索质量差就纠偏/联网补；
- **Self-RAG**：自己决定要不要检索、检索结果有没有用；
- **Adaptive RAG**：按问题类型路由到不同方案。

## 小结

- Agentic RAG = **决策循环**，检索成为模型可调用的“工具”；
- 控制好 **Stop Condition 与步数上限**是工程关键；
- 比固定流水线更鲁棒，但更贵、更难观测。